In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# input + select 9 MXenes

CSV_PATH = "./data/mxene_filtered.csv"  # your filtered 48 file
OUT_DIR = Path("cartoon")
OUT_DIR.mkdir(exist_ok=True)

SELECTED_9 = [
    "Y3C2O2",
    "Sc3C2O2",
    "Sc4C3O2",
    "Nb4C3O2H2",
    "Nb2CO2H2",
    "Mo2CO2H2",
    "Ta4C3O2H2",
    "Cr3C2F2",
    "Ta2CS2",
]

df = pd.read_csv(CSV_PATH)

# Sanity check: ensure all exist in the CSV
present = set(df["mxene"].astype(str))
missing = [m for m in SELECTED_9 if m not in present]
if missing:
    raise ValueError(f"These selected MXenes are missing from {CSV_PATH}: {missing}")
print("All 9 selected MXenes exist in the filtered CSV ✅")



All 9 selected MXenes exist in the filtered CSV ✅


In [2]:
# cartoon MXene builder (prototype-based, no DFT)

TRANSITION_METALS = {
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg"
}

# Rough in-plane lattice constants (Å) for visuals only
DEFAULT_A = {
    "Ti": 3.05, "V": 2.95, "Cr": 2.90, "Mo": 3.05, "W": 3.05,
    "Nb": 3.15, "Ta": 3.15, "Zr": 3.30, "Hf": 3.30, "Sc": 3.25, "Y": 3.60
}
def pick_a(metal: str) -> float:
    return DEFAULT_A.get(metal, 3.10)

def parse_formula(formula: str):
    tokens = re.findall(r"([A-Z][a-z]?)(\d*)", formula.strip())
    comp = {}
    for el, n in tokens:
        comp[el] = comp.get(el, 0) + (int(n) if n else 1)
    return comp

def build_hex_lattice(a: float):
    v1 = np.array([a, 0.0, 0.0])
    v2 = np.array([0.5*a, np.sqrt(3)/2*a, 0.0])
    return v1, v2

def frac_to_cart(frac, v1, v2):
    return frac[0]*v1 + frac[1]*v2 + np.array([0.0, 0.0, frac[2]])

def write_xyz(path, symbols, positions):
    with open(path, "w") as f:
        f.write(f"{len(symbols)}\n")
        f.write("cartoon MXene structure (prototype-based; not DFT-relaxed)\n")
        for s, (x,y,z) in zip(symbols, positions):
            f.write(f"{s:2s} {x: .6f} {y: .6f} {z: .6f}\n")

def build_mxene_cartoon(formula: str):
    comp = parse_formula(formula)

    # Identify metal M
    metals = [el for el in comp if el in TRANSITION_METALS]
    if not metals:
        raise ValueError(f"No transition metal found in {formula}")
    M = metals[0]
    nM = comp[M]

    # Identify X (C or N)
    nC = comp.get("C", 0)
    nN = comp.get("N", 0)
    if nC == 0 and nN == 0:
        raise ValueError(f"No C/N found in {formula}")
    X = "C" if nC >= nN else "N"

    # Terminations
    others = {el: cnt for el, cnt in comp.items() if el not in {M, "C", "N"}}

    # MXene prototypes
    if nM == 2:
        core_layers = ["M", "X", "M"]
    elif nM == 3:
        core_layers = ["M", "X", "M", "X", "M"]
    elif nM == 4:
        core_layers = ["M", "X", "M", "X", "M", "X", "M"]
    else:
        core_layers = ["M", "X", "M"]

    # Lattice
    a = pick_a(M)
    v1, v2 = build_hex_lattice(a)

    xy_A = (0.0, 0.0)
    xy_B = (1/3, 2/3)

    dz_MX = 1.05
    dz_MM = 0.90

    symbols = []
    positions = []   # <-- KEEP AS LIST UNTIL THE END

    def add_atom(el, z, xy):
        pos = frac_to_cart((xy[0], xy[1], z), v1, v2)
        symbols.append(el)
        positions.append(pos)

    # Build core
    z = 0.0
    toggle = 0
    for layer in core_layers:
        xy = xy_A if (toggle % 2 == 0) else xy_B
        if layer == "M":
            add_atom(M, z, xy)
            z -= dz_MM
        else:
            add_atom(X, z, xy)
            z -= dz_MX
        toggle += 1

    # Determine surface positions
    z_vals = [p[2] for p in positions]
    z_top = max(z_vals)
    z_bot = min(z_vals)

    # Termination placement
    term_sites = [xy_A, xy_B]
    z_term = 1.2
    z_H_from_O = 0.95

    nO = others.get("O", 0)
    nH = others.get("H", 0)

    if nO > 0 and nH > 0 and nO == nH:
        # (OH)n termination
        for xy in term_sites:
            add_atom("O", z_top + z_term, xy)
            add_atom("H", z_top + z_term + z_H_from_O, xy)
            add_atom("O", z_bot - z_term, xy)
            add_atom("H", z_bot - z_term - z_H_from_O, xy)
    else:
        # Simple termination (O, F, S, etc.)
        for el in others:
            if el == "H":
                continue
            for xy in term_sites:
                add_atom(el, z_top + z_term, xy)
                add_atom(el, z_bot - z_term, xy)

    # NOW convert to numpy array (safe)
    positions = np.array(positions)

    # Center structure in z
    z_center = 0.5 * (positions[:, 2].min() + positions[:, 2].max())
    positions[:, 2] -= z_center

    return symbols, positions


In [3]:
# export the 9 structures

for mx in SELECTED_9:
    sym, pos = build_mxene_cartoon(mx)
    out = OUT_DIR / f"{mx}_cartoon.xyz"
    write_xyz(out, sym, pos)
    print(f"Wrote: {out}")

print("\nDone. Open these .xyz files in VESTA / OVITO / ASE-GUI for quick visualization.")


Wrote: cartoon/Y3C2O2_cartoon.xyz
Wrote: cartoon/Sc3C2O2_cartoon.xyz
Wrote: cartoon/Sc4C3O2_cartoon.xyz
Wrote: cartoon/Nb4C3O2H2_cartoon.xyz
Wrote: cartoon/Nb2CO2H2_cartoon.xyz
Wrote: cartoon/Mo2CO2H2_cartoon.xyz
Wrote: cartoon/Ta4C3O2H2_cartoon.xyz
Wrote: cartoon/Cr3C2F2_cartoon.xyz
Wrote: cartoon/Ta2CS2_cartoon.xyz

Done. Open these .xyz files in VESTA / OVITO / ASE-GUI for quick visualization.
